# 08 — DrugGPT / GenMol: Small Molecule Candidate Generation

**Goal:** Generate novel small-molecule drug candidates targeting ERAP2.

- **DrugGPT** — GPT-based ligand generation, trained on 1.8M protein-ligand pairs
- **GenMol** — discrete diffusion for fragment-constrained molecule generation

**Run on:** Google Colab T4/V100 (8-12GB VRAM sufficient)

## References
- DrugGPT: https://huggingface.co/liyuesen/druggpt
- GenMol: https://huggingface.co/papers/2501.06158

In [ ]:
!pip install -q transformers torch rdkit-pypi

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load DrugGPT
model_name = "liyuesen/druggpt"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
model.eval()
print("DrugGPT loaded")

In [ ]:
# Generate molecules conditioned on ERAP2 binding pocket
# DrugGPT takes a protein pocket representation as input
# and generates SMILES strings for candidate ligands

# TODO: Extract ERAP2 binding pocket from PDB (notebook 05 output)
# The pocket residues are the zinc-binding site: H370, E371, H374, E393

def generate_candidates(pocket_input, n_candidates=100, temperature=0.8):
    """Generate SMILES candidates with DrugGPT."""
    candidates = []
    for i in range(n_candidates):
        inputs = tokenizer(pocket_input, return_tensors="pt").to("cuda")
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=temperature,
                do_sample=True,
                top_k=50,
            )
        smiles = tokenizer.decode(output[0], skip_special_tokens=True)
        candidates.append(smiles)
    return candidates

print("Upload ERAP2 pocket representation to generate candidates.")

In [ ]:
# Quick validation: check generated SMILES are valid with RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski

def validate_smiles(smiles_list):
    """Check validity and basic drug-likeness."""
    results = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            results.append({"smiles": smi, "valid": False})
            continue
        results.append({
            "smiles": smi,
            "valid": True,
            "mw": Descriptors.MolWt(mol),
            "logp": Descriptors.MolLogP(mol),
            "hbd": Descriptors.NumHDonors(mol),
            "hba": Descriptors.NumHAcceptors(mol),
            "lipinski_pass": Lipinski.NumHDonors(mol) <= 5 and
                            Lipinski.NumHAcceptors(mol) <= 10 and
                            Descriptors.MolWt(mol) <= 500 and
                            Descriptors.MolLogP(mol) <= 5,
        })
    return results